# 03 – Severity EDA (Claim Amount)

Purpose:
- Describe the **claim severity distribution** (`ClaimAmount`) on raw and log scales
- Quantify **tail heaviness** (high quantiles / max / top claims)
- Run quick integrity checks (nulls, non-positive values)
- Provide descriptive slices by key rating factors (for intuition, not modeling)

Data source:
- `data/staging/sev_train.parquet` (matched claims with policy features present)
- Optionally compares with `data/staging/sev_unmatched_claims.parquet` when available (claims excluded from training join)

Notes:
- Severity modeling is conditional on claim: `ClaimAmount > 0`.
- Because tails are extreme, prefer log-scale diagnostics and tail-focused evaluation.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "notebooks").exists():
            return p
    return start


ROOT = find_repo_root(Path.cwd())
os.chdir(ROOT)

STAGING_DIR = ROOT / "data" / "staging"
SEV_PATH = STAGING_DIR / "sev_train.parquet"
UNMATCHED_PATH = STAGING_DIR / "sev_unmatched_claims.parquet"

sev = pd.read_parquet(SEV_PATH)
unmatched = pd.read_parquet(UNMATCHED_PATH) if UNMATCHED_PATH.exists() else None

{
    "repo_root": str(ROOT),
    "sev_train_shape": tuple(sev.shape),
    "sev_unmatched_shape": None if unmatched is None else tuple(unmatched.shape),
}

(26444, 12)

In [ ]:
amt = pd.to_numeric(sev["ClaimAmount"], errors="coerce")

integrity = {
    "rows": int(len(sev)),
    "na_claimamount": int(amt.isna().sum()),
    "non_positive_claimamount": int((amt <= 0).sum(skipna=True)),
    "min": float(amt.min()),
    "max": float(amt.max()),
    "n_unique_policies": int(sev["IDpol"].nunique()) if "IDpol" in sev.columns else None,
}

# Claims per policy concentration (helps detect repeated small claims vs few huge losses)
if "IDpol" in sev.columns:
    c = sev["IDpol"].value_counts()
    integrity.update(
        {
            "claims_per_policy_mean": float(c.mean()),
            "claims_per_policy_p95": float(c.quantile(0.95)),
            "claims_per_policy_max": int(c.max()),
        }
    )

# Tail summary
q = [0.50, 0.90, 0.95, 0.99, 0.995, 0.999]
quantiles = {f"p{int(p*1000):03d}": float(amt.quantile(p)) for p in q}

summary = {
    "mean": float(amt.mean()),
    "median": float(amt.median()),
    **quantiles,
    "max": float(amt.max()),
}

{"integrity": integrity, "severity_summary": summary}

{'mean': 2265.5126493722582,
 'median': 1172.0,
 'p90': 2767.604,
 'p95': 4765.093499999995,
 'p99': 16451.223999999962,
 'p995': 34376.964349999944,
 'p999': 152223.24082000018,
 'max': 4075400.56}

In [ ]:
# Log-scale diagnostics (more stable under heavy tails)
amt_pos = amt[amt > 0].copy()
log_amt = np.log(amt_pos)

log_stats = {
    "log_mean": float(log_amt.mean()),
    "log_std": float(log_amt.std()),
    "log_p01": float(log_amt.quantile(0.01)),
    "log_p99": float(log_amt.quantile(0.99)),
}

# Top claims table (tail inspection)
top_claims = (
    sev.assign(ClaimAmount_num=amt)
    .sort_values("ClaimAmount_num", ascending=False)
    .head(10)
)

# Quick segment summaries (median and tail by rating factors)

def segment_summary(col: str, top_n: int = 10):
    if col not in sev.columns:
        return None
    df = sev.assign(ClaimAmount_num=amt)
    g = (
        df.groupby(col, dropna=False)
        .agg(
            rows=("ClaimAmount_num", "size"),
            mean=("ClaimAmount_num", "mean"),
            median=("ClaimAmount_num", "median"),
            p95=("ClaimAmount_num", lambda s: float(s.quantile(0.95))),
            p99=("ClaimAmount_num", lambda s: float(s.quantile(0.99))),
            max=("ClaimAmount_num", "max"),
        )
        .sort_values("rows", ascending=False)
        .head(top_n)
    )
    return g

{
    "log_stats": log_stats,
    "top_claims": top_claims[[c for c in ["IDpol", "ClaimAmount", "ClaimAmount_num", "Region", "VehBrand", "VehGas"] if c in top_claims.columns]],
    "by_region_top10": segment_summary("Region", top_n=10),
    "by_vehgas": segment_summary("VehGas", top_n=10),
}

{'log_mean': 6.8466766602503215,
 'log_std': 1.130022981743639,
 'log_p01': 3.6811751220328306,
 'log_p99': 9.708146630366855}

In [ ]:
# Optional: visualize severity distribution (log scale)
# If matplotlib is available in your environment, this provides a quick shape check.

try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8, 4))
    plt.hist(log_amt, bins=60)
    plt.title("log(ClaimAmount) histogram")
    plt.xlabel("log(ClaimAmount)")
    plt.ylabel("count")
    plt.tight_layout()
    plt.show()
except Exception as e:
    {"plot_skipped": True, "reason": str(e)[:200]}

### Conclusion

- Severity is **highly right-skewed with an extreme tail** (large gap between median, p99, and max). This makes mean-based summaries unstable and increases sensitivity to a small number of large losses.
- Log-scale summaries are much more stable; this supports a baseline severity model such as **Gamma GLM with log link** or **lognormal-style** modeling (depending on diagnostics).
- Next modeling-focused checks to add before fitting:
  - tail stability by segment (does p99 vary wildly by `Region`/`VehBrand`?)
  - consider a governed policy for tail handling (e.g., cap for monitoring vs uncapped for risk)
  - evaluate on both central and tail metrics (median/MAE on log-scale + p90/p95 stability)